In [1]:
import sys
sys.path.append('..')

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

# 1. import topological tool set
from padic_portfolio.topology.metric import correlation_to_distance, compute_mst, extract_ultrametric

#2. define mini-universe of stocks
tickers = ['AAPL', 'MSFT', 'XOM', 'CVX', 'JPM', 'GS', 'JNJ', 'PFE']

#3. download historical data (2 years worth)
print("Fetching market data...")
data = yf.download(tickers, start="2022-01-01", end="2024-01-01")['Close']

#4. calculate daily returns + correlation matrix
# drop first row b/c the first day has no "previous" day to calculate return from
returns = data.pct_change().dropna()
correlation_matrix = returns.corr()

#5. run topological pipeline
print("Runnin topological pipeline...")
dist_df = correlation_to_distance(correlation_matrix)
mst_matrix = compute_mst(dist_df)
ultrametric_df = extract_ultrametric(mst_matrix, labels=tickers)

#6. display p-adic skeleton
print("\n--- Subdominant Ultrametric Matrix ___")
display(ultrametric_df.round(3))

Fetching market data...


[*********************100%***********************]  8 of 8 completed

Runnin topological pipeline...

--- Subdominant Ultrametric Matrix ___


,AAPL,MSFT,XOM,CVX,JPM,GS,JNJ,PFE
AAPL,0.000,1.128,1.005,1.190,1.005,0.720,1.190,1.128
MSFT,1.128,0.000,1.128,1.190,1.128,1.128,1.190,0.516
XOM,1.005,1.128,0.000,1.190,0.727,1.005,1.190,1.128
CVX,1.190,1.190,1.190,0.000,1.190,1.190,1.042,1.190
JPM,1.005,1.128,0.727,1.190,0.000,1.005,1.190,1.128
GS,0.720,1.128,1.005,1.190,1.005,0.000,1.190,1.128
JNJ,1.190,1.190,1.190,1.042,1.190,1.190,0.000,1.190
PFE,1.128,0.516,1.128,1.190,1.128,1.128,1.190,0.000


In [3]:
from padic_portfolio.allocation.hrp import allocate_hrp

#run allocator using p-adic tree
final_weights = allocate_hrp(ultrametric_df, returns)

print("--- Final p-Adic Portfolio Weights ---")
display((final_weights * 100).round(2).astype(str) + '%')

--- Final p-Adic Portfolio Weights ---


CVX     34.13%
JNJ     15.85%
JPM      9.63%
MSFT     9.25%
XOM       8.8%
PFE       8.1%
AAPL     7.51%
GS       6.73%
dtype: object